# SSR vs CSR setups (to a single table)

Analysis based on the results from the K6 benchmark.

In [33]:
import pandas as pd;
import matplotlib.pyplot as plt
from sklearn.ensemble import IsolationForest

pd.set_option('display.float_format', lambda x: '%.2f' % x)
plt.rcParams.update({'font.size': 14})

path = '../../../k6/results/'

def convert_to_MiB(value):
    if 'GiB' in value:
        return str(int(float(value.replace('GiB', '')) * 1024))
    if 'MiB' in value:
        return value.replace('MiB', '')
    if 'B' in value:
        return str(int(float(value.replace('B', '')) / 1024))
    return value

def detect_outliers(df, features, contamination=0.1):
    clf = IsolationForest(contamination=contamination, random_state=42)
    outliers = clf.fit_predict(df[features])
    return outliers == 1

def graph_row(plot_func, dataset, features, height=3, width=20):
    cols = len(features)
    fig, axes = plt.subplots(ncols=cols, figsize=(width, height))
    plt.subplots_adjust(hspace=0.3, wspace=0.3, top=0.95, bottom=0.05)
    for x, f in enumerate(features):
        plot_func(dataset, f, axes[x], idx=x)
    return plt

services = ["monolith", "cdn", "discovery", "teasers", "recommendations"]

In [34]:
dirty_df = pd.read_csv(f'{path}monolith/10000/metrics.csv', sep=',')

services = ["monolith", "cdn"]
dirty_df['total_cpu_percent'] = dirty_df[[f'{s}_cpu_percent' for s in services]].sum(axis=1)
dirty_df['total_mem_usage'] = dirty_df[[f'{s}_mem_usage' for s in services]].sum(axis=1)

duration = ['duration_mean', 'duration_min', 'duration_max', 'duration_count']
throughput = ['throughput_mean', 'throughput_min', 'throughput_max', 'throughput_count']
err = ['err_mean', 'err_count']

dirty_df = dirty_df[dirty_df['timestamp'] >= 30].dropna(subset=[*duration, *throughput]).reset_index(drop=True)

mask = detect_outliers(dirty_df, [*duration, *throughput])
monolith_df = dirty_df[mask].copy().reset_index(drop=True)

In [35]:
dirty_df = pd.read_csv(f'{path}csr/10000/metrics.csv', sep=',')

services = ["monolith", "cdn"]
dirty_df['total_cpu_percent'] = dirty_df[[f'{s}_cpu_percent' for s in services]].sum(axis=1)
dirty_df['total_mem_usage'] = dirty_df[[f'{s}_mem_usage' for s in services]].sum(axis=1)

duration = ['duration_mean', 'duration_min', 'duration_max', 'duration_count']
throughput = ['throughput_mean', 'throughput_min', 'throughput_max', 'throughput_count']
err = ['err_mean', 'err_count']

dirty_df = dirty_df[dirty_df['timestamp'] >= 30].dropna(subset=[*duration, *throughput]).reset_index(drop=True)

mask = detect_outliers(dirty_df, [*duration, *throughput])
csr_df = dirty_df[mask].copy().reset_index(drop=True)

In [36]:
dirty_df = pd.read_csv(f'{path}ssrh/10000/metrics.csv', sep=',')

services = ["monolith", "cdn", "discovery", "teasers", "recommendations"]
dirty_df['total_cpu_percent'] = dirty_df[[f'{s}_cpu_percent' for s in services]].sum(axis=1)
dirty_df['total_mem_usage'] = dirty_df[[f'{s}_mem_usage' for s in services]].sum(axis=1)

duration = ['duration_mean', 'duration_min', 'duration_max', 'duration_count']
throughput = ['throughput_mean', 'throughput_min', 'throughput_max', 'throughput_count']
err = ['err_mean', 'err_count']

dirty_df = dirty_df[dirty_df['timestamp'] >= 30].dropna(subset=[*duration, *throughput]).reset_index(drop=True)

mask = detect_outliers(dirty_df, [*duration, *throughput])
ssrh_df = dirty_df[mask].copy().reset_index(drop=True)

In [37]:
dirty_df = pd.read_csv(f'{path}ssrv/10000/metrics.csv', sep=',')

services = ["monolith", "cdn", "discovery", "teasers", "recommendations", "homepage"]
dirty_df['total_cpu_percent'] = dirty_df[[f'{s}_cpu_percent' for s in services]].sum(axis=1)
dirty_df['total_mem_usage'] = dirty_df[[f'{s}_mem_usage' for s in services]].sum(axis=1)

duration = ['duration_mean', 'duration_min', 'duration_max', 'duration_count']
throughput = ['throughput_mean', 'throughput_min', 'throughput_max', 'throughput_count']
err = ['err_mean', 'err_count']

dirty_df = dirty_df[dirty_df['timestamp'] >= 30].dropna(subset=[*duration, *throughput]).reset_index(drop=True)

mask = detect_outliers(dirty_df, [*duration, *throughput])
ssrv_df = dirty_df[mask].copy().reset_index(drop=True)

In [43]:
rows = []

dfs = {
    'Monolith': monolith_df,
    'CSR': csr_df,
    'SSRH': ssrh_df,
    'SSRV': ssrv_df
}

for name, df in dfs.items():
    rows.append({
        "Setup": name,
        "Monolith":  f'{df["monolith_cpu_percent"].mean():.1f}%',
        "CDN":       f'{df["cdn_cpu_percent"].mean():.4f}%',
        "Discovery": f'{df["discovery_cpu_percent"].mean():.4f}%'             if 'discovery_cpu_percent' in df.columns else '-',
        "Teasers":   f'{df["teasers_cpu_percent"].mean():.3f}%'               if 'teasers_cpu_percent' in df.columns else '-',  
        "Recommendations": f'{df["recommendations_cpu_percent"].mean():.3f}%' if 'recommendations_cpu_percent' in df.columns else '-',
        "Homepage":  f'{df["homepage_cpu_percent"].mean():.1f}%'              if 'homepage_cpu_percent' in df.columns else '-'
    })
    
for name, df in dfs.items():
    rows.append({
        "Setup": name,
        "Monolith":  f'{df["monolith_mem_usage"].mean():.1f}MiB',
        "CDN":       f'{df["cdn_mem_usage"].mean():.1f}MiB',
        "Discovery": f'{df["discovery_mem_usage"].mean():.1f}MiB'             if 'discovery_mem_usage' in df.columns else '-',
        "Teasers":   f'{df["teasers_mem_usage"].mean():.1f}MiB'               if 'teasers_mem_usage' in df.columns else '-',  
        "Recommendations": f'{df["recommendations_mem_usage"].mean():.1f}MiB' if 'recommendations_mem_usage' in df.columns else '-',
        "Homepage":  f'{df["homepage_mem_usage"].mean():.1f}MiB'              if 'homepage_mem_usage' in df.columns else '-' 
    })
        

result_df = pd.DataFrame(rows)
result_df

,Setup,Monolith,CDN,Discovery,Teasers,Recommendations,Homepage
0,Monolith,118.8%,0.0015%,-,-,-,-
1,CSR,105.7%,0.0003%,0.0003%,-,-,-
2,SSRH,139.6%,0.0004%,3.7029%,35.352%,37.635%,-
3,SSRV,11.8%,0.0004%,3.2895%,0.003%,0.004%,73.3%
4,Monolith,4524.9MiB,47.4MiB,-,-,-,-
5,CSR,3884.6MiB,17.9MiB,17.7MiB,-,-,-
6,SSRH,4809.3MiB,17.5MiB,56.4MiB,476.1MiB,326.9MiB,-
7,SSRV,2690.0MiB,24.8MiB,66.6MiB,44.2MiB,41.8MiB,251.6MiB


In [53]:
rows = []

for name, df in dfs.items():
    rows.append({
        "Setup": name,
        "min":  f'{df["duration_min"].min():.1f}s',
        "mean":       f'{df["duration_mean"].mean():.2f}s',
        "max": f'{df["duration_max"].max():.1f}s',
        "throughput":   f'{df["throughput_mean"].mean():.1f} req/s',
        "failed": f'{int(df["err_count"].sum())}'
    })

result_df = pd.DataFrame(rows)
result_df

,Setup,min,mean,max,throughput,failed
0,Monolith,0.6s,0.83s,29.6s,166.7 req/s,0
1,CSR,0.4s,0.61s,12.6s,166.7 req/s,0
2,SSRH,4.0s,7.10s,91.0s,166.7 req/s,0
3,SSRV,4.9s,7.80s,19.5s,166.7 req/s,0
